# 06. Model Comparison and Transfer

A larger model is not automatically better. This module uses nested models and partial F tests to decide whether a block of predictors adds explanatory value beyond a reduced model.


In [ ]:
from lite_setup import ensure_packages
await ensure_packages()

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st
import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True


In [ ]:
delivery = pd.read_csv("data/delivery_time.csv")
full = smf.ols("Time ~ Cases + Distance", data=delivery).fit()
reduced = smf.ols("Time ~ Cases", data=delivery).fit()
comparison = sm.stats.anova_lm(reduced, full)
comparison


The reduced model must be nested inside the full model. Here, the reduced model removes `Distance` while keeping `Cases`. The partial F test asks whether distance adds value after cases are already in the model.


If the reduced model has $g$ predictors and the full model has $k$ predictors, the partial F statistic is

$$F_0=\frac{(SSE_R-SSE_F)/(k-g)}{SSE_F/(n-k-1)},$$

where $SSE_R$ is the reduced-model error sum of squares and $SSE_F$ is the full-model error sum of squares. Under the null hypothesis that the added predictors have zero coefficients, this follows an $F_{k-g,\,n-k-1}$ distribution.


In [ ]:
sse_reduced = np.sum(reduced.resid**2)
sse_full = np.sum(full.resid**2)
q = reduced.df_resid - full.df_resid
partial_f = ((sse_reduced - sse_full) / q) / (sse_full / full.df_resid)
p_value = 1 - st.f.cdf(partial_f, q, full.df_resid)

print(f"Manual partial F = {partial_f:.4f}")
print(f"p-value = {p_value:.6g}")


In [ ]:
print(f"Reduced adjusted R-squared: {reduced.rsquared_adj:.4f}")
print(f"Full adjusted R-squared:    {full.rsquared_adj:.4f}")


## Partial F Test for a Categorical Predictor Block

A categorical predictor with more than two levels creates a block of dummy variables. The right question is often whether the entire block improves the model, not whether one dummy coefficient happens to be significant by itself.


In [ ]:
stores = pd.read_csv("data/store_sales.csv")
reduced_location = smf.ols("SalesVolume ~ Households", data=stores).fit()
full_location = smf.ols('SalesVolume ~ Households + C(Location, Treatment(reference="Street"))', data=stores).fit()
sm.stats.anova_lm(reduced_location, full_location)


This test compares the model with household count only to the model that also includes the location dummy-variable block. It is the same logic as the partial F test above, applied to all location indicators together.


## Transfer Checklist

When you use multiple regression on a new problem, use this sequence:

1. Define the response and each predictor, including units.
2. Plot the response against each quantitative predictor and inspect predictor-predictor relationships.
3. Fit a model that matches the scientific question.
4. Interpret coefficients conditionally.
5. Use t tests for individual coefficients and F tests for joint questions.
6. Use confidence intervals for mean response and prediction intervals for future observations.
7. Prefer simpler models unless the added terms answer a real question or improve predictive performance for a defensible reason.


## Mini Transfer Exercise

Use the ozone dataset below. Fit a simple regression of ozone-warning days on the meteorological index, then ask what would change if you had a second meaningful predictor. The dataset itself only contains one predictor, so the goal is to practice defining the modeling question before adding variables.


In [ ]:
ozone = pd.read_csv("data/ozone_trends.csv")
ozone_model = smf.ols("OzoneDays ~ MeteorologicalIndex", data=ozone).fit()
print(ozone_model.summary())
